# Data Preprocessing Sanity check
## Choose dataset, preprocessing method, and review results

This notebook provides a unified interface to:
1. Load data
2. Choose a preprocessing method (basic, **Llama**, or **Gemini API**)
3. **Test with a small sample first** (NUM_SAMPLES = 5)
4. Review and compare results
5. Run on full dataset when satisfied

## Configuration

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import pprint
import google.generativeai as genai
from dotenv import load_dotenv

# Add project root to path
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)  
sys.path.append(project_root)
sys.path.append(os.path.join(project_root, "src"))  # Add src to path


try:
    from mosaic.preprocessing.translation_utils import (
        list_preprocessed_datasets,
        TruncationDiagnostic,
        compare_word_counts,
        compare_source_and_translated,
        show_preprocessing_stats,
        resolve_data_path
    )
except ImportError as e:
    print(f"Error loading modules: {e}")


print("Preprocessing module loaded successfully")

Preprocessing module loaded successfully


## Check inventory of available (preprocessed) datasets

In [2]:
# Auto-detect all available preprocessed data
inventory = list_preprocessed_datasets()

if "error" in inventory:
    print(f"Error: {inventory['error']}")
else:
    print(f"Found {len(inventory)} datasets with preprocessed files:\n")
    for name, data in inventory.items():
        print(f"{name} ({data['total_reports']} reports)")
        for f in data['files']:
            print(f"   - {f['filename']} ({f['size_mb']} MB) [{f['method'].upper()}]")
        print()

Found 6 datasets with preprocessed files:

5MeO_naturalistic (9197 reports)
   - 5MeO_naturalistic_preprocessed.csv (0.7 MB) [BASIC]

NDE (1457 reports)
   - NDE_preprocessed.csv (12.69 MB) [BASIC]

dreamachine_DL (98 reports)
   - dreamachine_DL_preprocessed.csv (0.03 MB) [BASIC]

dreamachine_HS (334 reports)
   - dreamachine_HS_preprocessed.csv (0.09 MB) [BASIC]

ganzfeld_GREEN (34 reports)
   - ganzfeld_GREEN_cleaned_llama.csv (0.05 MB) [LLAMA]

ganzfeld_RED (34 reports)
   - ganzfeld_RED_cleaned_llama.csv (0.06 MB) [LLAMA]



## Configutation: select Dataset and Preprocessing Method



In [3]:
# ========================================================
# CONFIGURATION
# ========================================================

# Paste the filename you want to check (e.g. 'MPE_cleaned_API.csv')
TARGET_FILE = 'NDE_preprocessed.csv'

# Column names (usually these defaults work)
SOURCE_COL = 'reflection_answer'
TARGET_COL = 'phen_report_english' #'cleaned_reflection'

# ========================================================

# FIX: Add 'DATA/' to the path string
full_path = resolve_data_path(f"DATA/preprocessed/{TARGET_FILE}")

if full_path.exists():
    print(f"Selected file: {TARGET_FILE}")
    print(f"   Path: {full_path}")
else:
    print(f"File not found: {TARGET_FILE}")
    print(f"   Checked path: {full_path}")
    print("Check the filename from the list above.")

Selected file: NDE_preprocessed.csv
   Path: /Users/rb666/Projects/MOSAIC/DATA/preprocessed/NDE_preprocessed.csv


In [4]:
#Checks row counts and looks for an associated error log.
show_preprocessing_stats(str(full_path))


PREPROCESSING STATISTICS

Loading preprocessed data from: /Users/rb666/Projects/MOSAIC/DATA/preprocessed/NDE_preprocessed.csv
Loaded 1457 rows
Reports (rows):        1457
Note: No 'sentences' column found (run basic_preprocess first)

No error log found




In [5]:
#check truncation: Calculates retention rates and checks for cut-off text.
diagnostic = TruncationDiagnostic(
    str(full_path), 
    source_col=SOURCE_COL, 
    target_col=TARGET_COL
)

diagnostic.run_full_diagnostic()


DATA TRUNCATION DIAGNOSTIC REPORT

File: /Users/rb666/Projects/MOSAIC/DATA/preprocessed/NDE_preprocessed.csv
Rows: 1457
Columns: ['Unnamed: 0', 'reflection_answer', 'Language', 'phen_report_english']

1. ERROR MARKERS (Processed with Errors)
----------------------------------------------------------------------------------------------------
✅ No error markers detected.

2. TOKEN LIMIT ANALYSIS (LLM max_tokens setting)
----------------------------------------------------------------------------------------------------
LLM max_tokens setting: 8192
Estimated max chars:    32768

Texts at risk (>90% of limit):  13 (0.9%)
Texts exceeding limit:          0
Max source text size:           32767 chars
Estimated max source tokens:    8192 tokens

✅ Token limits seem adequate

3. DATA RETENTION ANALYSIS
----------------------------------------------------------------------------------------------------
Overall word retention rate:  98.2%
Total words lost:            22,596

Distribution of rete

In [6]:
#visual comaparison of translation vs original
compare_source_and_translated(
    diagnostic.df, 
    source_col=SOURCE_COL, 
    target_col=TARGET_COL, 
    num_samples=3,
    anonymise=False  # Set to True to mask names/PII in output
)


COMPARISON: reflection_answer → phen_report_english (3 examples)

[Row 238]
SOURCE (1966 chars):
En mai 1987 à l’hôpital Sainte Elisabeth de Namur, par le docteur Cuisinier, opéré pour la deuxième fois du grand carrefour (pose d’une prothèse, des artères fémorales jusqu’à la ?). Au moment de me réveiller, j’avais des douleurs des deux côtés de la cage thoracique, je suppose qu’on m’avait choqué
...

TARGET (1841 chars):
In May 1987 at the Sainte Elisabeth hospital in Namur, I was operated on for the second time by Dr. Cuisinier, with a prosthesis being placed from the femoral arteries to ?. When I woke up, I had pains on both sides of my thoracic cage, I suppose I was shocked to make me come back ! And it was then 
...
----------------------------------------------------------------------------------------------------

[Row 974]
SOURCE (8438 chars):
On April 9, 1992, I was in the midst of a heated argument with my 15-year-old daughter.  I don’t even remember what it was about.  All I 